# Лекция 12. Обучение с подкреплением (RL)

## Мотивация

<video width="100%" controls>
  <source src="../data/open-ai.mp4" type="video/mp4">
  Ваш браузер не поддерживает видео.
</video>

Привет! Вот и подходит к концу последняя информативная пара на нашем курсе 2025 - 2026 уч. года. Сегодня мы немного отойдем в сторону от обучения с учителем (`supervised learning`). Сегодня мы поговорим об обучении с подкреплением (`reinforcement learning`).


До сих пор сам процесс **обучения** во всех алгоритмах, которые мы изучили, был основан на том, что имеется некоторая **обучающая выборка** (мы её часто задавали в виде `dataset`). Стоит сказать, что в академических целях мы использовали всегда готовые размеченные данные, а вот в реальной жизни это далеко не так: либо данных у нас нет, либо нам нужно их ещё собрать и разметить.

Но что если мы откажемся от огромных датасетов и попробуем вести обучение модели при помощи метода **проб и ошибок** (`trial and error`)? **Обучением с подкреплением** (`RL`) как раз этим и занимается!

Но начнем с того, что вспомним основную постановку задачи в обучении с учителем (`supervised learning`):

### Дано:

* Объекты и ответы $x \in \mathbb{X}$ и $y \in \mathbb{Y}$
* Loss-функция $L(\hat{y}, y)$ (Обычно $L$ - гладкая или хотя бы дифф.)
* Семейство алгоритмов (моделей): $a \in \mathbb{A}, \ a: \mathbb{X} \to \mathbb{Y}$ 

### Цель:
* Необходимо найти такой оптимальный $a^*$, что: 
$$a^* = \arg\min_{a} L(a(x), y)$$


А вот так устроено **Обучение с подкреплением** (`RL`):

### Дано:

* Объекты ~~и ответы~~ $x \in \mathbb{X}$
* Функция качества $L(\hat{y})$ (Обычно $L$ сложно формулируемая, не дифф.)
* Семейство алгоритмов (моделей): $a \in \mathbb{A}, \ a: \mathbb{X} \to \mathbb{Y}$ 

### Цель:
* Необходимо найти такой оптимальный $a^*$, что: 
$$a^* = \arg\max_{a} L(a(x))$$


![alt](../data/run.gif)

Какую фукнцию награды можно задать для такой модели?

## Основные понятия

Давайте введём основные определения, с которыми мы будем работать. Сразу же оговоримся, что эта лекция является вводной в `RL`. Мы не будем рассматривать какие-то продвинутые техники, а обсудим основные идеи и векторы развития.

![env-state-agent-reward](../data/env_agent.png)

Задача `RL` задаётся **Марковским процессом принятия решений** (`MDP`) - это четверка $(\mathcal{S}, \mathcal{A}, \mathcal{P}, r)$, где:
* $\mathcal{S}$ - пространство состояний (`state space`), множество состояний, в которых в каждый момент времени может находиться среда.
* $\mathcal{A}$ - пространство действий (`action space`), множество вариантов, из которых нужно производить выбор на каждом шаге своего взаимодействия со средой.
* $\mathcal{P}$ - функция переходов (transition function), которая задаёт изменение среды после того, как в состоянии $s \in \mathcal{S}$ было выбрано действие $a \in \mathcal{A}$. 
    * В общем случае функция переходов может быть стохастична, и тогда такая функция переходов моделируется распределением $p(s'|s, a)$: с какой вероятностью в какое состояние перейдет среда после выбора действия $a$ в состоянии $s$.
* $r: \mathcal{S} \times \mathcal{A} \to \mathbb{R}$ - функция награды (`reward function`), выдающая скалярную величину за выбор действия $a$ в состоянии $s$. Это "обучающий сигнал".


![alt](../data/autonomouslearninggif-5.gif)

Задача агента — заработать как можно большую награду:
1. Либо за отведенное время $h$ (от слова horizon, «горизонт»); такая постановка задачи называется моделью с конечным горизонтом, и целевую функцию (доход, revenue) можно представить так:
$$R = \mathbb{E}[r_0 + r_1 + ... + r_h] = \mathbb{E}[\sum_{t=0}^h r_t],$$

где $r_t$ - награда агента на $t$-ом шаге.

2. Либо за бесконечное время. В таких задачах принято следующее правило: **получить награду раньше выгоднее, чем позже**, поэтому в задачах с **бесконечным горизонтом** вводят величину $\gamma$ - `discount factor`, на которую награда уменьшается на очередном шаге:

$$R = \mathbb{E}[r_0 + \gamma r_1 + \gamma^2 r_2 + ... + \gamma^k r_k + ...] = \mathbb{E}[{\sum_{t=0}^{\infty} \gamma^t r_t}]$$


Введём ещё несколько понятий - **агента** (`agent`) и **политику** (`policy`):

Традиционно субъект, взаимодействующий со средой и влияющий на неё, называется в обучении с подкреплением **агентом** (`agent`). Агент руководствуется некоторым правилом, возможно, тоже стохастичным, как выбирать действия в зависимости от текущего состояния среды, которое называется **политика** (`policy`) и моделируется распределением $\pi (a | s)$

Именно политику мы и будем с вами искать, аналогично тому, как в `SL` мы ищем какую-то функцию.

### Моделирование взаимодействия агента с политикой $\pi$ со средой

Взаимодействие со средой агента с политикой $\pi (a | s)$ моделируется так:
1. Изначально среда находится в состоянии $s_0$
2. Агент сэмплирует действие из своей политики $a_0 \sim \pi (a_0 | s_0)$
3. Среда отвечает на это: 
    * сэмплирует своё следующее состояние $s_1 \sim p(s_1 | s_0, a_0)$
    * Выдаёт агенту награду в размере $r(s_0, a_0)$
4. Процесс повторяется...


Так повторяется до бесконечности, либо пока среда не перейдет в терминальное состояние (в нём взаимодействие обрывается). Если в среде есть терминальные состояния, то один проход от начального состояния $s_0$ до некоторого терминального состояния $s_T$ называется **сессией** (`episode` или `session`).

А цепочка случайных величин $s_0, a_0, s_1, a_1, ...$ называется тректорией (`trajectory`)

![mario-handbook-ml](../data/mario.webp)


Итак, фактически среда для нас — это управляемая марковская цепь: на каждом шаге выбором $a \in \mathcal{A}$ определяем то распределение, из которого будет генерироваться следующее состояние. Мы предполагаем:

1. Во-первых, марковское свойство: что переход в следующее состояние определяется лишь текущим состоянием и не зависит от всей предыдущей истории:

$$p(s_{t+1} | s_t, a_t, s_{t-1}, a_{t-1}, ..., s_0, a_0) = p(s_{t+1} | s_t, a_t)$$

2. Во-вторых, мы предполагаем стационарность: функция переходов $p(s' | s, a)$ не зависит от времени, от того, сколько шагов прошло с начала взаимодействия. Это довольно реалистичные предположения: законы мира не изменяются со временем (стационарность), а состояние — описывает мир целиком (марковость). 

## Задача о многоруком бандите

![alt](../data/multi_armed.png)

Самая простая постановка задачи обучения с подкреплением — это так называемая задача о многоруких бандитах (multiarmed bandits). Формально здесь все точно так же, но $|S| = 1$, то есть состояние агента не меняется. У него просто есть некий фиксированный набор действий $\mathcal{A}$ и возможность выбирать из этого набора действий. 

Здесь возникает одно из основных диллем в `RL`:

## Exploration vs Exploitation

**Exploration** (Исследование) — агент совершает действия, которые могут быть неоптимальными с точки зрения текущей награды, чтобы собрать новую информацию о среде (о переходах между состояниями, о скрытых наградах).

**Exploitation** (Использование) — агент выбирает действие, которое на данный момент считается наилучшим (жадное действие) на основе текущей оцененной функции ценности.

## Связь с реальными когнитивными процессами

![alt](../data/torn_dog.png)

![alt](../data/torn_cat.jpg)

![alt](../data/rat.webp)

![alt](../data/reflect.webp)

## Cross-Entropy Method

Наша основная задача: максимизировать нашу награду $\mathbb{E}_{\pi}[R]$ меняя политику $\pi(a | s)$

Как нам это сделать? У нас же нет никаких градиентов!

Тогда будем действовать так:

1. Иницилизируем $\pi$ случайным образом
2. Сэмплируем $N$ сессий при помощи политики $\pi$
3. Выбираем $M$ элитных сессий (`elite`) среди всех $N$ сессий
4. Новая политика $\pi_new$ не полностью случайна. Повторяем процесс..

Такой метод называется методом `Cross-Entropy`
Для простоты, считаем, что $\mathcal{S}$ и $\mathcal{A}$ конечны.

![heatmap](../data/heatmap.png)


Окей, мы можем заключить, что в таком случае политике $\pi(a|s)$ соответсвует некоторая матрица $A_{s, a}$

Теперь некоторые вещи более формально:

* Сэмплируем $N$ сессий с политикой $\pi$
* Выбираем $M$ элитных сессий (`elite`) среди всех $N$ сессий:
$$Elite = [(s_0, a_0), (s_1, a_1), ..., (s_M, a_M)]$$

* Обновляем политику:
$$\pi_{new}(a|s) = \frac{\sum_{s_t, a_t \in Elite} [s_t = s] [a_t = a]}{\sum_{s_t, a_t \in Elite} [s_t = s]}$$


$$\pi_{new}(a|s) = \frac{Сколько \ раз  \ мы  \ совершили \ действие  \ a \ в \ состоянии \ s}{Сколько \ раз \ мы  \ были \ в \ состоянии \ s}$$

Мы придумали, как нам выучить оптимальное поведение, получая информацию от среды, не имея никаких градиентных методов оптимизации. Мы только что перешли к задаче `SL`: мы учимся для состояний предсказывать те действия $a$, которые были в элитных сессиях $Elite$. Мы породили обучающую выборку, взаимодействуя со средой. ($s$ - это наш объекта, а $a$ - это наш ответ)


Если у нас есть возможность разметить выборку, то никто не мешает нам решать более сложные задачи (если состояния непрерывны, а действия дискретны).

![doom](../data/doom.jpg)

Давайте вспомним, а что вообще делает политика $\pi(a|s)$? Она задаёт вероятностное расапределение действий, при условии состояния. А что может задавать распределение дискретной конечной величины, при условии картинки? Любой классификатор, который принимает на вход картинку:

* $\pi(a|s) = f_w(a, s)$  ($f$, к примеру, может быть, `RFC` или `LR` или `NN` и т.д.)
* Сэмплируем $N$ сессий с политикой $\pi$. Выбираем $M$ элитных сессий 
$$Elite = [(s_0, a_0), (s_1, a_1), ..., (s_M, a_M)]$$

* Дообучаем наш классификатор, максимизируя правдоподобие:

$$\pi_{new}(a|s) =  \arg\max_{\pi}\sum_{s_t, a_t \in Elite} log \ \pi(a_i | s_i))$$


### Континуальный случай
А что делать если $\mathcal{S}$ и $\mathcal{A}$ континуальны?

Но мы ведь умеем предсказывать непрерывные величины. Да, возьмем регрессор (или несколько регрессоров)

* $$ \pi(a | s) = \mathcal{N}(\mu_w(a, s), \sigma_{\gamma}^2(a, s))$$
* Сэмплируем $N$ сессий с политикой $\pi$. Выбираем $M$ элитных сессий 
$$Elite = [(s_0, a_0), (s_1, a_1), ..., (s_M, a_M)]$$

* Дообучаем наши регрессоры, максимизируя правдоподобие:

$$max_{w, \gamma} \sum_{s_t, a_t \in Elite} log\  \mathcal{N}(\mu_w(a, s), \sigma_{\gamma}^2(a, s))\$$

$$\pi_{new}(a|s) =  \arg\max_{\pi}\sum_{s_t, a_t \in Elite} log \ \pi(a_i | s_i))$$

## Основные отличия DL от SL

| Аспект | Supervised Learning (SL) | Reinforcement Learning (RL) |
|--------|--------------------------|-----------------------------|
| **Данные** | Независимые примеры `(x, y)` | Последовательность `(s, a, r, s')` |
| **Обратная связь** | Немедленная (на каждый x есть y) | Отложенная (reward приходит позже) |
| **Цель** | Минимизировать ошибку на тесте | Максимизировать суммарную награду |
| **Распределение данных** | Фиксированное (i.i.d.) | Зависит от действий агента (non-i.i.d.) |
| **Сбор данных** | Пассивный (датасет дан) | Активный (agent сам собирает) |
| **Exploration** | Не нужен | Критически важен |
| **Credit assignment** | Тривиален (каждый x → свой y) | Сложен (какое действие привело к награде?) |
| **Стационарность** | Данные стационарны | Среда может меняться |

<video width="100%" controls>
  <source src="../data/open-ai.mp4" type="video/mp4">
  Ваш браузер не поддерживает видео.
</video>

## Немного об обучении роботов

![alt](../data/rover.gif)

1. Эффективность сэмплирования

2. Безопасность
- RL пробует случайные действия (exploration)
- Робот может:
  - Упасть и сломаться
  - Повредить окружающих
  - Испортить оборудование
- Компании не могут позволить себе «эпизод с падением робота»

3. Reward engineering
- Как задать награду для «идти красиво»?
- Награда за скорость → робот бежит и падает
- Штраф за падение → робот стоит на месте


Нас ограничивает физика реального мира.